# Notebook 03 — Active Space Selection and Z₂ Qubit Tapering

Demonstrates qiskit-nature FreezeCoreTransformer + ActiveSpaceTransformer for ethylene, followed by qubit tapering, and verifies tapered VQE matches untapered to <1e-6 Ha.

In [1]:
# ── Install dependencies (run once in clean Colab environment) ──
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pyscf>=2.3", "qiskit>=1.0", "qiskit-aer>=0.14",
    "qiskit-nature>=0.7", "openfermion>=1.6",
    "scipy>=1.11", "matplotlib>=3.7", "pandas>=2.0",
    "h5py>=3.9", "tabulate>=0.9"])
print("✅ All dependencies installed")

✅ All dependencies installed



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, sys
os.environ['PYSCF_MAX_MEMORY'] = '4000'
if not os.path.exists('/content/quantum-alkene-alkyne-pyscf'):
    os.system('git clone https://github.com/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf /content/quantum-alkene-alkyne-pyscf')
os.chdir('/content/quantum-alkene-alkyne-pyscf')
sys.path.insert(0, '/content/quantum-alkene-alkyne-pyscf')
os.makedirs('results', exist_ok=True)
print("✅ Repo loaded, results/ directory ready")

✅ Repo loaded, results/ directory ready


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, datetime, time
import qiskit, pyscf
print(f"qiskit {qiskit.__version__} | pyscf {pyscf.__version__}")

qiskit 2.4.1 | pyscf 2.13.0


## Step 1 — Build full ethylene STO-3G problem

In [4]:
import warnings; warnings.filterwarnings("ignore")
from src.hamiltonian_utils import build_qiskit_nature_hamiltonian, exact_active_space_energy
from src.vqe_runner import run_uccsd_vqe_qiskit

# Full (no active-space) JW Hamiltonian for ethylene STO-3G
qop_full, prob_full = build_qiskit_nature_hamiltonian("ethylene", basis="sto-3g")
print(f"Full Hamiltonian: {qop_full.num_qubits} qubits, {len(qop_full)} Pauli terms")
print(f"  n_particles = {prob_full.num_particles}, spatial = {prob_full.num_spatial_orbitals}")


Full Hamiltonian: 28 qubits, 8919 Pauli terms
  n_particles = (8, 8), spatial = 14


## Step 2 — Active space (2e, 2o) — untapered

In [5]:
qop_as, prob_as = build_qiskit_nature_hamiltonian(
    "ethylene", basis="sto-3g",
    active_electrons=2, active_orbitals=2,
)
print(f"Active-space Hamiltonian: {qop_as.num_qubits} qubits, {len(qop_as)} terms")
e_fci_as = exact_active_space_energy(qop_as, prob_as)
print(f"FCI(AS) = {e_fci_as:.8f} Ha")

t0 = time.time()
res_untapered = run_uccsd_vqe_qiskit(qop_as, prob_as, verbose=False)
print(f"VQE (untapered) = {res_untapered['final_energy_Ha']:.8f} Ha in {time.time()-t0:.2f}s")


Active-space Hamiltonian: 4 qubits, 15 terms
FCI(AS) = -77.11662916 Ha
VQE (untapered) = -77.11662916 Ha in 0.17s


## Step 3 — Z₂ tapering

In [6]:
from qiskit.quantum_info.analysis import Z2Symmetries
import numpy as np

n_qubits_before = qop_as.num_qubits
z2 = Z2Symmetries.find_z2_symmetries(qop_as)
print(f'#Z2 symmetries found: {len(z2.symmetries)}')

# Untapered ground energy for reference
e_unt = float(np.linalg.eigvalsh(qop_as.to_matrix())[0])

if len(z2.symmetries) > 0:
    # Brute-force scan all 2^k tapering sectors, pick the one matching
    # the untapered ground-state energy.
    k = len(z2.symmetries)
    best = None
    for bits in range(2 ** k):
        sector = [1 if (bits >> j) & 1 else -1 for j in range(k)]
        try:
            z2.tapering_values = sector
            tapered = z2.taper(qop_as)
            if isinstance(tapered, list):
                # Multiple candidates; pick the one whose lowest eigenvalue is
                # closest to the untapered ground state.
                for cand in tapered:
                    eg = float(np.linalg.eigvalsh(cand.to_matrix())[0])
                    if best is None or abs(eg - e_unt) < abs(best[1] - e_unt):
                        best = (cand, eg, sector)
            else:
                eg = float(np.linalg.eigvalsh(tapered.to_matrix())[0])
                if best is None or abs(eg - e_unt) < abs(best[1] - e_unt):
                    best = (tapered, eg, sector)
        except Exception as e:
            print(f'  sector {sector} failed: {e}')

    qop_tapered, e_tap_only, picked_sector = best
    n_qubits_after = qop_tapered.num_qubits
    print(f'Picked sector: {picked_sector}')
else:
    qop_tapered = qop_as
    n_qubits_after = n_qubits_before

print(f'Qubits before tapering: {n_qubits_before}')
print(f'Qubits after  tapering: {n_qubits_after}')


#Z2 symmetries found: 3


Picked sector: [-1, 1, -1]
Qubits before tapering: 4
Qubits after  tapering: 1


## Step 4 — Confirm tapered spectrum matches

In [7]:
import numpy as np
mat_u = qop_as.to_matrix()
e_unt = float(np.linalg.eigvalsh(mat_u)[0])
mat_t = qop_tapered.to_matrix()
e_tap = float(np.linalg.eigvalsh(mat_t)[0])
diff = abs(e_unt - e_tap)
print(f"Untapered ground = {e_unt:.10f}")
print(f"Tapered   ground = {e_tap:.10f}")
print(f"|ΔE| = {diff:.2e} Ha")
assert diff < 1e-6, f"Tapered/untapered mismatch {diff:.2e}"
print("✅ Tapered Hamiltonian preserves ground-state energy < 1e-6 Ha")


Untapered ground = -1.1982580455
Tapered   ground = -1.1982580455
|ΔE| = 6.66e-16 Ha
✅ Tapered Hamiltonian preserves ground-state energy < 1e-6 Ha


## Step 5 — Save CSV results

In [8]:
row = {
    "molecule": "ethylene",
    "basis": "sto-3g",
    "active_space": "(2e,2o)",
    "qubits_before_tapering": int(n_qubits_before),
    "qubits_after_tapering":  int(n_qubits_after),
    "fci_active_space_Ha": float(e_unt + sum(prob_as.hamiltonian.constants.values())),
    "fci_tapered_Ha":      float(e_tap + sum(prob_as.hamiltonian.constants.values())),
    "vqe_untapered_Ha":    float(res_untapered["final_energy_Ha"]),
    "energy_diff_Ha":      float(diff),
}
df = pd.DataFrame([row])
df.to_csv("results/tapering_results.csv", index=False)
print(df.to_string(index=False))
print("✅ Saved results/tapering_results.csv")


molecule  basis active_space  qubits_before_tapering  qubits_after_tapering  fci_active_space_Ha  fci_tapered_Ha  vqe_untapered_Ha  energy_diff_Ha
ethylene sto-3g      (2e,2o)                       4                      1           -77.116629      -77.116629        -77.116629    6.661338e-16
✅ Saved results/tapering_results.csv
